# Time-Series Noise Demo — Visual Sampling of Temporal Variability

This notebook analyzes real machine-vibration and PV-power sessions. The key scientific question is **which portions of a 128-sample signal are visually inspected before the participant classifies variability as LOW, MEDIUM, or HIGH**.

`tobii-pytracker` is used for session loading, fixation detection, scanpath analysis, and two-dimensional gaze entropy. Sample-level AOI scoring is performed from the recorded `timeseries_bboxes` stored in `objects_bboxes`; the current public `BBoxAttentionAnalyzer` targets `image_bboxes`, so sample AOIs require a small experiment-specific hit-test layer.

## Scientific context and experimental logic

This notebook implements the analysis rationale described in [Time-Series Noise Demo](../../../docs/basic_examples/timeseries_noise_demo.md). The experiment asks how observers inspect **structured numeric signals** while judging short-term variability, rather than treating the plots as ordinary static images.

### Experimental structure

The canonical dataset contains 18 fixed time-series stimuli split across two signal domains:

| Domain | LOW | MEDIUM | HIGH | Total |
|---|---:|---:|---:|---:|
| Machine vibration | 3 | 3 | 3 | 9 |
| PV power | 3 | 3 | 3 | 9 |
| **Total** | **6** | **6** | **6** | **18** |

Each trial retains 128 ordered samples and a target class (`LOW`, `MEDIUM`, or `HIGH`). The experiment stores per-sample spatial regions in `timeseries_bboxes`, allowing attention to be expressed in the coordinate system of the underlying signal instead of only in screen pixels.

### Scientific model

Time-series classification requires evidence integration across the horizontal axis. The relevant question is therefore not only *how much* the participant looked, but **which temporal regions were sampled, how broadly the signal was covered, and whether previously inspected regions were revisited**.

The notebook intentionally maintains two complementary coordinate systems:

- **screen-space measures** from built-in analyzers, such as fixation, scanpath, and spatial entropy;
- **signal-index measures** derived from the recorded sample bboxes, such as inspected sample span, segment coverage, and 1-D temporal entropy.

| Documentation hypothesis | Operational measure in this notebook | Primary interpretation |
|---|---|---|
| H1 — Variability and effort | fixation count/duration by LOW/MEDIUM/HIGH | Does visual inspection effort change with signal variability? |
| H2 — Middle-category ambiguity | trial duration, revisits, segment entropy for MEDIUM | Is the intermediate category visually harder to classify? |
| H3 — Exploration | unique samples, sample span, segment coverage | Does higher variability produce broader temporal sampling? |
| H4 — Domain effect | machine vs PV summaries within class | Does the same judgment use different visual strategies across domains? |
| H5 — Behavioral difficulty | accuracy by class | Is MEDIUM behaviorally less discriminable than LOW/HIGH? |

### Measurement caution

A fixation assigned to a sample bbox indicates that the fixation centroid overlapped the recorded region for that sample. It does **not** prove that the exact numeric value was cognitively processed. Likewise, broader coverage or higher entropy should be described as a gaze-distribution property, not automatically as higher cognitive load or lower confidence.

The two domains are presented as separate blocks in the current demo. Consequently, domain comparisons are descriptive because domain is potentially confounded with block order, practice, or fatigue. A confirmatory study would counterbalance or randomize block order across participants.

The demo documentation cites Duchowski (2017) for general eye-tracking methodology.


## 1. Analysis parameters

This cell controls fixation extraction and the temporal segmentation used for signal-index analysis. Keeping these parameters explicit is essential because the interpretation spans two levels: screen-space gaze events and sample-index AOIs.

The 8-segment representation is a descriptive coarse-graining of the 128 samples. It is intended to stabilize temporal attention profiles, not to redefine the underlying signal.


In [ ]:
SELECTED_SESSIONS = {"machine": None, "pv": None}
FIXATION_PARAMS = {"method":"dispersion", "dispersion_threshold":50.0, "min_duration":0.10}
N_TEMPORAL_SEGMENTS = 8
SPATIAL_ENTROPY_BINS = 40
RUN_FIXATION_SENSITIVITY = False
SENSITIVITY_DISPERSION_THRESHOLDS = [40.0, 50.0, 60.0]


## 2. Load real sessions for both domains

Machine-vibration and PV-power data are collected as separate domain sessions. The notebook loads whichever valid sessions exist and preserves `domain` as an explicit factor before combining trial-level results.

No synthetic replacement is created when one domain is missing. Partial collection can therefore be inspected without being mistaken for the full 18-trial design.

> **tobii-pytracker support:** Each session is loaded through `CustomConfig` and `DataLoader`, so the notebook works with the same output contract as the rest of the library.


In [ ]:
from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tobii_pytracker.configs.custom_config import CustomConfig
from tobii_pytracker.analyze import (
    DataLoader,
    FixationAnalyzer,
    ScanpathsAnalyzer,
)


def find_demo_root() -> Path:
    """Locate tobii-pytracker-demo from common Jupyter launch locations."""
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for candidate in candidates:
        if (candidate / "examples").is_dir() and (candidate / "output").is_dir():
            return candidate
        nested = candidate / "tobii-pytracker-demo"
        if (nested / "examples").is_dir() and (nested / "output").is_dir():
            return nested
    raise FileNotFoundError("Could not locate the tobii-pytracker-demo repository root.")


def newest_subject(loader: DataLoader) -> str:
    subjects = loader.get_subjects()
    if not subjects:
        raise FileNotFoundError(f"No experiment sessions found under {loader.output_root}")
    def mtime(subject: str) -> float:
        return (loader.output_root / subject / "data.csv").stat().st_mtime
    return max(subjects, key=mtime)


def prepare_session(config_path: Path, subject: str | None = None):
    """Load one real session with DataLoader; never synthesize replacement data."""
    config = CustomConfig(str(config_path))
    loader = DataLoader(config=config, root=DEMO_ROOT)
    selected = subject or newest_subject(loader)
    raw = loader.get_subject_data(selected, flatten=False).reset_index(drop=True)
    raw.insert(0, "set_name", selected)
    raw["slide_index"] = np.arange(len(raw), dtype=int)
    flat = loader.get_subject_data(selected, flatten=True)
    return loader, selected, raw, flat


def safe_parse(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    text = "" if value is None else str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default


def point_in_centered_bbox(x: float, y: float, bbox: dict, margin: float = 2.0) -> bool:
    try:
        cx, cy, w, h = (float(bbox[k]) for k in ("cx", "cy", "w", "h"))
    except (KeyError, TypeError, ValueError):
        return False
    return (cx - w/2 - margin <= x <= cx + w/2 + margin and
            cy - h/2 - margin <= y <= cy + h/2 + margin)


def normalize_token(value) -> str:
    return re.sub(r"[^0-9a-ząćęłńóśźż]+", "", str(value).casefold())


def trial_gaze_counts(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    counts = pd.Series(0, index=range(n_trials), dtype=int)
    if not flat.empty and {"slide_index", "avg_gaze_x"}.issubset(flat.columns):
        observed = flat.dropna(subset=["avg_gaze_x", "avg_gaze_y"]).groupby("slide_index").size()
        for idx, count in observed.items():
            if int(idx) in counts.index:
                counts.loc[int(idx)] = int(count)
    return counts


def trial_observed_duration(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    duration = pd.Series(np.nan, index=range(n_trials), dtype=float)
    if not flat.empty and {"slide_index", "system_time"}.issubset(flat.columns):
        for idx, group in flat.dropna(subset=["system_time"]).groupby("slide_index"):
            if len(group) >= 2 and int(idx) in duration.index:
                duration.loc[int(idx)] = float(group["system_time"].max() - group["system_time"].min())
    return duration


def run_fixations(flat: pd.DataFrame, output_dir: Path, params: dict) -> pd.DataFrame:
    required = {"set_name", "slide_index", "avg_gaze_x", "avg_gaze_y", "system_time"}
    if flat.empty or not required.issubset(flat.columns):
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    clean = flat.dropna(subset=["avg_gaze_x","avg_gaze_y","system_time"]).copy()
    if clean.empty:
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    analyzer = FixationAnalyzer(output_dir, **params)
    return analyzer.analyze(clean)


def run_scanpaths(fixations: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    if fixations.empty:
        return pd.DataFrame(columns=["set_name","slide_index","distance"])
    return ScanpathsAnalyzer(output_dir).analyze(fixations, per="slide")


def summarize_fixations(fixations: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if fixations.empty:
        return base.assign(fixation_count=0, total_fixation_duration=0.0, mean_fixation_duration=np.nan)
    agg = (fixations.groupby("slide_index")
           .agg(fixation_count=("duration","size"),
                total_fixation_duration=("duration","sum"),
                mean_fixation_duration=("duration","mean"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"fixation_count":0,"total_fixation_duration":0.0})


def summarize_scanpaths(scanpaths: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if scanpaths.empty:
        return base.assign(scanpath_transition_count=0, scanpath_distance=0.0)
    agg = (scanpaths.groupby("slide_index")
           .agg(scanpath_transition_count=("distance","size"), scanpath_distance=("distance","sum"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"scanpath_transition_count":0,"scanpath_distance":0.0})

DEMO_ROOT = find_demo_root()
print(f"Demo repository: {DEMO_ROOT}")

from tobii_pytracker.analyze import EntropyAnalyzer

In [ ]:
example_dir = DEMO_ROOT / "examples" / "timeseries_noise_demo"
config_candidates = sorted(example_dir.glob("config*.yaml"))
configs = {domain: next(p for p in config_candidates if domain in p.stem) for domain in ("machine", "pv")}
datasets = {"machine": pd.read_csv(example_dir/"data/machine_vibration.csv"),
            "pv": pd.read_csv(example_dir/"data/pv_power.csv")}

sessions = {}
for domain, config_path in configs.items():
    try:
        loader, session, raw, flat = prepare_session(config_path, SELECTED_SESSIONS[domain])
    except FileNotFoundError as exc:
        warnings.warn(str(exc)); continue
    sessions[domain] = {"loader":loader,"session":session,"raw":raw,"flat":flat}

if not sessions:
    raise FileNotFoundError("No machine or PV sessions were found. Run the experiment before analysis.")
print({domain: bundle["session"] for domain,bundle in sessions.items()})

## 3. Reproducibility record and design integrity

Each domain session is fingerprinted separately, and the expected 9-trial 3/3/3 class balance is checked before cross-domain comparison. This prevents a partially collected block from silently changing class proportions.

The design audit also records which fixed CSV fixture was used, which is important because signal-index metrics refer directly to those 128 committed samples.


In [ ]:
import hashlib
import platform
from importlib.metadata import PackageNotFoundError, version as package_version


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def installed_version(distribution: str) -> str:
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return "package-metadata-unavailable"


def provenance_table(session: str, data_csv: Path, dataset_path: Path, analysis_label: str) -> pd.DataFrame:
    record = {
        "analysis_label": analysis_label,
        "session": str(session),
        "data_csv_sha256": sha256_file(data_csv),
        "dataset_sha256": sha256_file(dataset_path),
        "python": platform.python_version(),
        "tobii_pytracker": installed_version("tobii-pytracker"),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }
    return pd.DataFrame([record])


def save_json(path: Path, payload: dict):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

provenance_rows=[]; design_rows=[]
for domain,bundle in sessions.items():
    dataset_path=example_dir/"data"/("machine_vibration.csv" if domain=="machine" else "pv_power.csv")
    data_csv=bundle["loader"].output_root/bundle["session"]/"data.csv"
    record=provenance_table(bundle["session"],data_csv,dataset_path,f"timeseries_{domain}_v2").iloc[0].to_dict()
    record["domain"]=domain; provenance_rows.append(record)
    counts=bundle["raw"]["classification"].astype(str).str.casefold().value_counts().to_dict()
    design_rows.extend([
        {"domain":domain,"check":"9 recorded trials","passed":len(bundle["raw"])==9,"observed":len(bundle["raw"]),"expected":9},
        {"domain":domain,"check":"LOW/MEDIUM/HIGH balance","passed":counts=={"low":3,"medium":3,"high":3},"observed":str(counts),"expected":"low=3, medium=3, high=3"},
    ])
provenance=pd.DataFrame(provenance_rows); design_checks=pd.DataFrame(design_rows)
display(provenance); display(design_checks)
if not design_checks.empty and not bool(design_checks["passed"].all()): warnings.warn("At least one available domain session is incomplete or non-canonical.")

## 4. Data-quality audit and built-in event extraction

The first analytical layer is ordinary eye-movement quality and event extraction. Behavioral classifications remain available even when gaze is missing; fixation, scanpath, and entropy metrics are computed only on usable gaze.

Built-in analyzers describe exploration in screen coordinates. Later sections translate fixation locations into the more scientifically meaningful sample-index coordinate system.

> **tobii-pytracker support:** Fixation and scanpath events are derived with `FixationAnalyzer` and `ScanpathsAnalyzer`; screen-space dispersion is additionally summarized with `EntropyAnalyzer`.


In [ ]:
def stimulus_id(value) -> str:
    name = Path(str(value).replace("\\", "/")).name
    return name[:-4] if name.lower().endswith(".png") else name


def sample_boxes(objects_bboxes):
    objects = safe_parse(objects_bboxes, dict, {})
    return objects.get("timeseries_bboxes", []) if isinstance(objects, dict) else []


def assign_fixations_to_samples(raw: pd.DataFrame, fixations: pd.DataFrame) -> pd.DataFrame:
    records=[]
    if fixations.empty:
        return pd.DataFrame(columns=["set_name","slide_index","sample_index","duration","fix_start","fixation_order"])
    for _,trial in raw.iterrows():
        slide=int(trial["slide_index"]); boxes=sample_boxes(trial.get("objects_bboxes"))
        f=fixations[fixations["slide_index"]==slide].sort_values("fix_start").reset_index(drop=True)
        for order,fix in f.iterrows():
            hits=[]
            for rec in boxes:
                bbox=rec.get("bbox",{}) if isinstance(rec,dict) else {}
                if point_in_centered_bbox(fix["x_mean"],fix["y_mean"],bbox):
                    try: hits.append(int(rec.get("start_idx")))
                    except (TypeError,ValueError): pass
            if hits:
                records.append({"set_name":trial["set_name"],"slide_index":slide,"sample_index":hits[0],
                                "duration":float(fix["duration"]),"fix_start":float(fix["fix_start"]),"fixation_order":int(order)})
    return pd.DataFrame(records)

def segment_entropy(sample_hits: pd.DataFrame, segments: int = 8) -> pd.DataFrame:
    rows = []
    if sample_hits.empty:
        return pd.DataFrame(columns=["domain", "set_name", "slide_index", "temporal_entropy"])
    width = 128 // segments
    data = sample_hits.copy()
    data["segment"] = (data["sample_index"] // width).clip(upper=segments - 1)
    group_cols = ["set_name", "slide_index"]
    if "domain" in data.columns:
        group_cols = ["domain"] + group_cols
    for keys, g in data.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        key_map = dict(zip(group_cols, keys))
        weights = g.groupby("segment")["duration"].sum().reindex(range(segments), fill_value=0.0).to_numpy(float)
        if weights.sum() <= 0:
            ent = np.nan
        else:
            p = weights / weights.sum()
            p = p[p > 0]
            ent = float(-(p * np.log2(p)).sum())
        rows.append({**key_map, "temporal_entropy": ent})
    return pd.DataFrame(rows)


In [ ]:
all_trials=[]; all_fix=[]; all_scan=[]; all_entropy=[]; all_sample_hits=[]
analysis_dirs={}
for domain,b in sessions.items():
    raw=b["raw"].copy(); flat=b["flat"].copy(); loader=b["loader"]; session=b["session"]
    raw["domain"] = domain
    raw["stimulus_id"] = raw["screenshot_file"].map(stimulus_id)
    meta = datasets[domain].copy(); meta["stimulus_id"] = meta["id"].astype(str)
    raw = raw.merge(meta[["stimulus_id","class"]], on="stimulus_id", how="left", suffixes=("","_dataset"))
    raw["expected_class"] = raw["classification"].astype(str).str.casefold()
    raw["response"] = raw["user_classification"].astype(str).str.casefold()
    raw["correct"] = raw["response"] == raw["expected_class"]
    raw["uncertain"] = raw["response"].isin(["none", "i don't know", "nie wiem"])
    counts=trial_gaze_counts(flat,len(raw)); dur=trial_observed_duration(flat,len(raw))
    raw["gaze_samples"] = raw["slide_index"].map(counts); raw["usable_gaze"] = raw["gaze_samples"]>0
    raw["observed_gaze_duration_s"] = raw["slide_index"].map(dur)
    out = loader.output_root / session / "analysis_timeseries_demo_v2"; out.mkdir(exist_ok=True); analysis_dirs[domain]=out
    fix=run_fixations(flat,out,FIXATION_PARAMS); scan=run_scanpaths(fix,out)
    clean_flat = flat.dropna(subset=["avg_gaze_x","avg_gaze_y","system_time"]).copy() if not flat.empty else pd.DataFrame()
    entropy = EntropyAnalyzer(out).analyze(clean_flat, per="slide", bins=SPATIAL_ENTROPY_BINS) if not clean_flat.empty else pd.DataFrame()
    hits=assign_fixations_to_samples(raw,fix)
    if not fix.empty: fix=fix.assign(domain=domain)
    if not scan.empty: scan=scan.assign(domain=domain)
    if not entropy.empty: entropy=entropy.assign(domain=domain)
    if not hits.empty: hits=hits.assign(domain=domain)
    all_trials.append(raw); all_fix.append(fix); all_scan.append(scan); all_entropy.append(entropy); all_sample_hits.append(hits)

raw_all=pd.concat(all_trials,ignore_index=True)
fixations=pd.concat(all_fix,ignore_index=True) if all_fix else pd.DataFrame()
scanpaths=pd.concat(all_scan,ignore_index=True) if all_scan else pd.DataFrame()
spatial_entropy=pd.concat(all_entropy,ignore_index=True) if all_entropy else pd.DataFrame()
sample_hits=pd.concat(all_sample_hits,ignore_index=True) if all_sample_hits else pd.DataFrame()
display(raw_all[["domain","slide_index","stimulus_id","expected_class","response","correct","gaze_samples","usable_gaze"]])

## 5. Optional fixation-parameter sensitivity

When enabled, this diagnostic re-runs the built-in fixation detector independently for each available signal domain.

In [ ]:
sensitivity_rows=[]
if RUN_FIXATION_SENSITIVITY:
    for domain,b in sessions.items():
        for threshold in SENSITIVITY_DISPERSION_THRESHOLDS:
            params=dict(FIXATION_PARAMS); params["dispersion_threshold"]=float(threshold)
            detected=run_fixations(b["flat"],analysis_dirs[domain]/"sensitivity",params)
            sensitivity_rows.append({"domain":domain,"dispersion_threshold":float(threshold),"fixation_count":len(detected),
                                     "mean_fixation_duration":float(detected["duration"].mean()) if not detected.empty else np.nan})
sensitivity=pd.DataFrame(sensitivity_rows)
if not sensitivity.empty: display(sensitivity)
else: print("Sensitivity analysis is disabled or no domain data are available.")

## 6. Sample-level attention and temporal coverage

Every stored `timeseries_bbox` links a spatial region to a range of signal samples. Fixation centroids are assigned to those recorded regions, producing a direct mapping from visual attention to signal index.

The notebook then derives unique sample coverage, sample span, first/last attended sample, weighted attention location, 8-segment coverage, revisits, and 1-D temporal entropy. These metrics address **where along the temporal sequence evidence was sampled**, which is the central scientific motivation of the demo.

The 1-D entropy reported here is intentionally distinct from the built-in 2-D gaze entropy: one summarizes distribution over temporal segments, the other summarizes spatial distribution on the screen.

> **tobii-pytracker support:** The `timeseries_bboxes` used here are written by the time-series experiment. Mapping fixation centroids to sample indices is demo-specific notebook logic because the current public bbox-attention analyzer does not directly score `timeseries_bboxes`.


In [ ]:
def sample_sequence_summary(group: pd.DataFrame) -> pd.Series:
    g=group.sort_values("fix_start").copy()
    sequence=g["sample_index"].astype(int).tolist()
    segments=(g["sample_index"]//(128//N_TEMPORAL_SEGMENTS)).clip(upper=N_TEMPORAL_SEGMENTS-1).astype(int).tolist()
    seen=set(); last=None; revisits=0
    for value in segments:
        if value != last and value in seen:
            revisits += 1
        seen.add(value); last=value
    dwell_by_segment=(g.assign(segment=segments).groupby("segment")["duration"].sum()) if len(g) else pd.Series(dtype=float)
    return pd.Series({
        "unique_samples":len(set(sequence)),
        "sample_min":min(sequence) if sequence else np.nan,
        "sample_max":max(sequence) if sequence else np.nan,
        "sample_span":max(sequence)-min(sequence) if sequence else np.nan,
        "first_sample":sequence[0] if sequence else np.nan,
        "last_sample":sequence[-1] if sequence else np.nan,
        "mean_attended_sample":float(np.average(g["sample_index"],weights=g["duration"])) if g["duration"].sum()>0 else np.nan,
        "attended_fixations":len(g),
        "attended_dwell_s":float(g["duration"].sum()),
        "unique_segments":len(set(segments)),
        "segment_revisits":revisits,
        "most_attended_segment":int(dwell_by_segment.idxmax()) if not dwell_by_segment.empty else np.nan,
    })

trial_keys=raw_all[["domain","set_name","slide_index"]].copy()
if sample_hits.empty:
    sample_summary=trial_keys.assign(unique_samples=np.nan,sample_min=np.nan,sample_max=np.nan,sample_span=np.nan,first_sample=np.nan,last_sample=np.nan,
                                     mean_attended_sample=np.nan,attended_fixations=np.nan,attended_dwell_s=np.nan,unique_segments=np.nan,segment_revisits=np.nan,most_attended_segment=np.nan,temporal_entropy=np.nan)
else:
    sample_rows=[]
    for (domain, set_name, slide_index), group in sample_hits.groupby(["domain","set_name","slide_index"]):
        record=sample_sequence_summary(group).to_dict()
        sample_rows.append({"domain":domain,"set_name":set_name,"slide_index":int(slide_index),**record})
    sample_summary=pd.DataFrame(sample_rows)
    sample_summary=sample_summary.merge(segment_entropy(sample_hits,N_TEMPORAL_SEGMENTS),on=["domain","set_name","slide_index"],how="left")

fix_sum=[]; scan_sum=[]
for domain,b in sessions.items():
    n=len(b["raw"])
    fs=summarize_fixations(fixations[fixations["domain"]==domain] if "domain" in fixations else pd.DataFrame(),n).assign(domain=domain)
    ss=summarize_scanpaths(scanpaths[scanpaths["domain"]==domain] if "domain" in scanpaths else pd.DataFrame(),n).assign(domain=domain)
    fix_sum.append(fs); scan_sum.append(ss)
fix_summary=pd.concat(fix_sum,ignore_index=True); scan_summary=pd.concat(scan_sum,ignore_index=True)

trial_metrics=(raw_all[["domain","set_name","slide_index","stimulus_id","expected_class","response","correct","uncertain","gaze_samples","usable_gaze","observed_gaze_duration_s"]]
               .merge(fix_summary,on=["domain","slide_index"],how="left")
               .merge(scan_summary,on=["domain","slide_index"],how="left")
               .merge(sample_summary,on=["domain","set_name","slide_index"],how="left"))
if not spatial_entropy.empty:
    trial_metrics=trial_metrics.merge(spatial_entropy[["domain","set_name","slide_index","entropy","convex_hull_area"]],on=["domain","set_name","slide_index"],how="left")
trial_metrics["sample_coverage"]=trial_metrics["unique_samples"]/128.0
trial_metrics["segment_coverage"]=trial_metrics["unique_segments"]/float(N_TEMPORAL_SEGMENTS)
eye_cols=["fixation_count","total_fixation_duration","mean_fixation_duration","scanpath_transition_count","scanpath_distance","unique_samples","sample_span","first_sample","last_sample","mean_attended_sample","attended_fixations","attended_dwell_s","unique_segments","segment_revisits","most_attended_segment","temporal_entropy","sample_coverage","segment_coverage","entropy","convex_hull_area"]
existing=[c for c in eye_cols if c in trial_metrics.columns]
trial_metrics.loc[~trial_metrics["usable_gaze"],existing]=np.nan
display(trial_metrics)

## 7. Hypothesis-oriented summaries

The summary tables are aligned with the five hypotheses in the demo documentation:

1. LOW/MEDIUM/HIGH fixation effort for H1.
2. MEDIUM trial duration, revisit behavior, and temporal entropy for H2.
3. sample coverage and span for H3.
4. domain-by-class comparison for H4.
5. behavioral accuracy by variability class for H5.

Domain should be treated cautiously because machine and PV are separate blocks. Any apparent domain effect may contain order, learning, or fatigue effects unless block order is counterbalanced in a future study.


In [ ]:
summary=(trial_metrics.groupby(["domain","expected_class"])
 .agg(n_trials=("slide_index","size"),accuracy=("correct","mean"),uncertainty_rate=("uncertain","mean"),usable_gaze_rate=("usable_gaze","mean"),
      mean_observed_gaze_duration_s=("observed_gaze_duration_s","mean"),mean_fixation_count=("fixation_count","mean"),mean_fixation_duration=("mean_fixation_duration","mean"),
      mean_scanpath_distance=("scanpath_distance","mean"),mean_sample_coverage=("sample_coverage","mean"),mean_sample_span=("sample_span","mean"),
      mean_segment_revisits=("segment_revisits","mean"),mean_temporal_entropy=("temporal_entropy","mean"),mean_spatial_entropy=("entropy","mean")).reset_index())
display(summary)

confusion=pd.crosstab([trial_metrics["domain"],trial_metrics["expected_class"]],trial_metrics["response"],dropna=False)
display(confusion)

## 8. Temporal attention profiles

These profiles aggregate attention over equal-width temporal segments. They provide an interpretable summary of whether observers concentrated on early, middle, or late portions of the signal and whether that pattern changes with class or domain.

A broad profile indicates distributed sampling; it does not automatically imply uncertainty. Interpret it together with trial duration, revisits, classification accuracy, and the underlying signal shape.


In [ ]:

segment_profile=pd.DataFrame()
if not sample_hits.empty:
    tmp=sample_hits.copy(); tmp["segment"]=(tmp["sample_index"]//(128//N_TEMPORAL_SEGMENTS)).clip(upper=N_TEMPORAL_SEGMENTS-1)
    labels=raw_all[["domain","set_name","slide_index","expected_class"]]
    tmp=tmp.merge(labels,on=["domain","set_name","slide_index"],how="left")
    segment_profile=tmp.groupby(["domain","expected_class","segment"],as_index=False)["duration"].sum()
    segment_profile["dwell_share"]=segment_profile["duration"]/segment_profile.groupby(["domain","expected_class"])["duration"].transform("sum")
    fig,axes=plt.subplots(1,max(1,len(segment_profile["domain"].unique())),figsize=(12,4),squeeze=False)
    for ax,(domain,gdomain) in zip(axes.flat,segment_profile.groupby("domain")):
        for label,g in gdomain.groupby("expected_class"):
            ax.plot(g["segment"],g["dwell_share"],marker="o",label=label)
        ax.set(xlabel="Temporal segment (0=earliest)",ylabel="Fixation-dwell share",title=f"{domain}: temporal attention")
        ax.legend()
    plt.tight_layout(); plt.show()

fig,axes=plt.subplots(2,2,figsize=(12,9))
summary.pivot(index="expected_class",columns="domain",values="accuracy").plot(kind="bar",ax=axes[0,0]); axes[0,0].set_ylim(0,1); axes[0,0].set_title("Classification accuracy")
summary.pivot(index="expected_class",columns="domain",values="mean_fixation_count").plot(kind="bar",ax=axes[0,1]); axes[0,1].set_title("Fixation count")
summary.pivot(index="expected_class",columns="domain",values="mean_sample_coverage").plot(kind="bar",ax=axes[1,0]); axes[1,0].set_title("Sample-index coverage")
summary.pivot(index="expected_class",columns="domain",values="mean_temporal_entropy").plot(kind="bar",ax=axes[1,1]); axes[1,1].set_title("1-D temporal entropy")
plt.tight_layout(); plt.show()


## 9. Inspect one representative signal trial

This section links the abstract sample-index metrics back to the original signal fixture. Overlaying attended regions on the actual 128-sample trace helps verify that the AOI mapping is plausible and reveals which peaks, troughs, or intervals contributed to visual inspection.

The figure is descriptive: attention to a sample region does not prove that the participant encoded the exact numerical magnitude.

> **tobii-pytracker support:** The underlying gaze/fixation data come from the standard pytracker output and analyzers; the signal-plus-attention overlay itself is a task-specific Matplotlib visualization.


In [ ]:

def signal_values(domain: str, stimulus_id: str):
    frame=datasets[domain]; row=frame.loc[frame["id"].astype(str)==str(stimulus_id)]
    if row.empty: return None
    value_columns=list(frame.columns[1:-1])
    return row.iloc[0][value_columns].astype(float).to_numpy()

representative_record=None
available=trial_metrics.dropna(subset=["gaze_samples"]).sort_values("gaze_samples",ascending=False)
if not available.empty:
    representative_record=available.iloc[0]
    domain=str(representative_record["domain"]); slide=int(representative_record["slide_index"]); stimulus=str(representative_record["stimulus_id"])
    values=signal_values(domain,stimulus)
    if values is not None:
        hits=sample_hits[(sample_hits["domain"]==domain)&(sample_hits["slide_index"]==slide)].sort_values("fix_start") if not sample_hits.empty else pd.DataFrame()
        fig,ax=plt.subplots(figsize=(12,4)); ax.plot(np.arange(len(values)),values,linewidth=1.5)
        if not hits.empty:
            idx=hits["sample_index"].astype(int).clip(0,len(values)-1).to_numpy()
            ax.scatter(idx,values[idx],s=np.maximum(30,hits["duration"].to_numpy()*250),alpha=0.65)
        ax.set(xlabel="Sample index",ylabel="Signal value",title=f"{domain} / {stimulus}: signal with attended samples")
        plt.show()
    b=sessions[domain]
    if representative_record["gaze_samples"]>0:
        b["loader"].plot_gaze(b["session"],slide,gradient=True,show=True)
else:
    print("No usable gaze is available for representative-trial visualization.")


## 10. Export derived results

Derived trial-level, class-level, domain-level, and temporal-profile tables are exported without modifying the raw experiment output. Analysis parameters and provenance should travel with these tables because fixation thresholds and segmentation choices affect the derived measures.

For group studies, add participant identifiers before combining sessions and retain stimulus IDs so domain, class, participant, and item effects remain distinguishable.


In [ ]:

combined_root=DEMO_ROOT/"output"/"timeseries_noise_demo"/"analysis_latest_v2"; combined_root.mkdir(parents=True,exist_ok=True)
trial_metrics.to_csv(combined_root/"trial_metrics.csv",index=False)
summary.to_csv(combined_root/"domain_class_summary.csv",index=False)
confusion.to_csv(combined_root/"response_confusion.csv")
sample_hits.to_csv(combined_root/"sample_fixation_hits.csv",index=False)
segment_profile.to_csv(combined_root/"segment_attention_profile.csv",index=False)
fixations.to_csv(combined_root/"fixations.csv",index=False); scanpaths.to_csv(combined_root/"scanpaths.csv",index=False)
provenance.to_csv(combined_root/"provenance.csv",index=False); design_checks.to_csv(combined_root/"design_checks.csv",index=False)
if not sensitivity.empty: sensitivity.to_csv(combined_root/"fixation_sensitivity.csv",index=False)
save_json(combined_root/"analysis_parameters.json",{"fixation":FIXATION_PARAMS,"temporal_segments":N_TEMPORAL_SEGMENTS,"spatial_entropy_bins":SPATIAL_ENTROPY_BINS,"sensitivity_thresholds":SENSITIVITY_DISPERSION_THRESHOLDS})
print(combined_root)


## 11. Scientific interpretation and limitations

The demo documentation motivates the analysis as a study of **evidence sampling along an ordered signal**, not merely gaze on a picture of a graph. The notebook therefore distinguishes screen-space exploration from signal-index exploration throughout.

- Trials without gaze retain their behavioral classification but contribute no eye-movement metrics.
- Sample coverage means that fixation centroids overlapped recorded sample regions; it does not prove processing of exact values.
- MEDIUM-category revisits or entropy are compatible with ambiguity but are not direct confidence measures.
- Broader temporal coverage can reflect useful evidence integration, uncertainty, or stimulus complexity.
- Machine and PV are separate blocks, so domain is confounded with block/order in the current demonstration.
- The LOW/MEDIUM/HIGH labels are properties of the fixed fixtures, not universal signal thresholds.
- Confirmatory work should counterbalance block order, include more stimuli and participants, preregister event parameters, and model participant/stimulus variability.

**Documentation link:** `docs/basic_examples/timeseries_noise_demo.md`.
